# Homework 2 — Market Data Cleaning and Validation

**Author:** Sterling Couts  
**Course:** UC Berkeley pre-MFE Python  
**Objective:** Assess the quality of the supplied daily equity data and produce a validated dataset for reproducible stock risk reports.

## Data sources and observation structure

The source is the course-provided SQLite database, `data.db`.

- **`ohlc_hw`** contains the market data to inspect and clean.
- **`ohlc`** is a reference table that may support trading-date checks and date-specific price-adjustment factors.

Each observation is intended to represent one security on one date, identified by `(ticker, ts)`. Identifier consistency and key uniqueness must be verified before time-series calculations.

## Data-handling approach

Preserve the source database and retain an unchanged in-memory copy of the loaded homework data. Perform repairs on a separate dataset, document the reasons for changes, and retain repair flags. Write the validated table to a separate homework database copy.

A statistical outlier is a review flag, not proof of an error. Any unresolved observations will be reported explicitly rather than repaired without a defensible basis.

## 1. Environment and source location

Confirm the Python interpreter, notebook working directory, and source database location before loading data. These checks help distinguish environment or path problems from problems in the dataset.

In [1]:
from pathlib import Path
import sqlite3
import sys

import pandas as pd

# Locate the course checkout without relying on a personal Windows path.
course_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "Lectures" / "Lecture 2" / "hw2.md").is_file()),
    None,
)
if course_root is None:
    raise FileNotFoundError("Run this notebook from your homework folder in the course repository.")

source_db = course_root / "Lectures" / "Lecture 4" / "data" / "data.db"

print("Python interpreter:", sys.executable)
print("Python version:", sys.version.split()[0])
print("pandas version:", pd.__version__)
print("Working directory:", Path.cwd())
print("Source database:", source_db)

if not source_db.is_file():
    raise FileNotFoundError(f"Source database not found: {source_db}")

print("Source file check: passed")

Python interpreter: C:\Users\scouts.admin\Documents\UCB-MFE-python-preprogram-2026-winter\.venv\Scripts\python.exe
Python version: 3.14.8
pandas version: 2.3.3
Working directory: C:\Users\scouts.admin\Documents\UCB-MFE-python-preprogram-2026-winter\Homeworks\HW2\Sterling_Couts\work\final_verification
Source database: C:\Users\scouts.admin\Documents\UCB-MFE-python-preprogram-2026-winter\Lectures\Lecture 4\data\data.db
Source file check: passed


**Initial setup finding:** The source database file exists at the configured location. The interpreter, package version, and working directory are recorded above.

This check establishes file availability only. Table structure, data completeness, and economic validity remain to be assessed.

## 2. Load the original homework data

**Purpose:** Load the full `ohlc_hw` table for initial inspection, retaining the supplied observations and column values.

Open the source database in read-only mode and explicitly close the connection after loading. Parse `ts` as a date column to support chronological inspection and subsequent time-series operations.

No filtering, deduplication, identifier normalization, or price repair is performed at this stage. The resulting DataFrame, `raw`, will serve as the unchanged baseline for comparisons with the cleaned dataset.

**Initial checks:** Inspect the row count, column types, nonmissing counts, summary statistics, date range, and raw ticker labels. Distinct raw labels will not yet be interpreted as distinct securities.

### Column dictionary

| Column | Meaning |
|---|---|
| `ts` | Observation date; interpreted as a trading-session date. |
| `ticker` | Vendor security label; normalize formatting before grouping. |
| `open` | Supplied unadjusted opening price for the session. |
| `high` | Supplied unadjusted highest price during the session. |
| `low` | Supplied unadjusted lowest price during the session. |
| `close` | Supplied unadjusted closing price for the session. |
| `adj_close` | Vendor-adjusted closing price under its adjustment convention. |
| `volume` | Reported trading volume for the session; missing means unknown, not zero. |

The assignment calls OHLC raw prices. We use that convention without independently certifying how the vendor handled historical corporate actions. Adjusted close is not constrained by the raw daily high-low range.


In [2]:
from contextlib import closing

source_uri = source_db.resolve().as_uri() + "?mode=ro"

with closing(sqlite3.connect(source_uri, uri=True)) as conn:
    raw = pd.read_sql_query(
        "SELECT * FROM ohlc_hw",
        conn,
        parse_dates=["ts"],
    )

display(raw.head())

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


**Initial observation:** The preview contains the expected date, ticker, OHLC, adjusted-close, and volume fields. The displayed rows span multiple securities and are not chronologically ordered. Time-series operations will therefore require explicit sorting by normalized ticker and date.

Adjusted close differs from raw close in the displayed observations. This difference alone does not establish a data-quality issue; adjusted-price validity and consistency will be assessed separately.

The five-row preview does not establish completeness, key uniqueness, or validity of the full dataset.

In [3]:
raw.info()
display(raw.describe())

print("Number of rows:", len(raw))
print("Earliest observation:", raw["ts"].min())
print("Latest observation:", raw["ts"].max())
print("Distinct raw ticker labels:", raw["ticker"].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


Number of rows: 397711
Earliest observation: 1962-01-02 00:00:00
Latest observation: 2024-12-06 00:00:00
Distinct raw ticker labels: 150


### Initial inspection findings

The delivered homework table contains 397,711 rows across eight columns, with observation dates ranging from January 2, 1962 through December 6, 2024. These are overall bounds and do not establish continuous or equal coverage across securities.

Adjusted close contains 19,855 missing values, approximately 4.99% of delivered rows. All other columns have complete non-null counts. Non-null values may still be invalid, and these counts do not detect absent trading-date rows.

The minimum raw opening price is zero, indicating at least one violation of the required positive-price condition. Zero trading volume and a large maximum adjusted close also warrant observation-level review; their causes have not yet been established.

There are 150 distinct raw ticker labels. This count should not yet be interpreted as the number of securities because label formatting has not been normalized.

All results above describe the original delivery before normalization, deduplication, or repair.

In [4]:
raw_ticker_labels = sorted(raw["ticker"].unique())

for label in raw_ticker_labels:
    print(repr(label))

'<a target="_blank"><span style="color: blue">AAPL</span></a>'
'<a target="_blank"><span style="color: blue">AMGN</span></a>'
'<a target="_blank"><span style="color: blue">AMZN</span></a>'
'<a target="_blank"><span style="color: blue">AXP</span></a>'
'<a target="_blank"><span style="color: blue">BA</span></a>'
'<a target="_blank"><span style="color: blue">CAT</span></a>'
'<a target="_blank"><span style="color: blue">CRM</span></a>'
'<a target="_blank"><span style="color: blue">CSCO</span></a>'
'<a target="_blank"><span style="color: blue">CVX</span></a>'
'<a target="_blank"><span style="color: blue">DIS</span></a>'
'<a target="_blank"><span style="color: blue">GS</span></a>'
'<a target="_blank"><span style="color: blue">HD</span></a>'
'<a target="_blank"><span style="color: blue">HON</span></a>'
'<a target="_blank"><span style="color: blue">IBM</span></a>'
'<a target="_blank"><span style="color: blue">JNJ</span></a>'
'<a target="_blank"><span style="color: blue">JPM</span></a>'
'<a tar

### Ticker-label inspection

The raw label list contains plain ticker symbols and HTML-wrapped variants with different color and font-weight attributes. These presentation differences cause the same ticker text to appear as multiple distinct labels.

Normalization will remove HTML markup, trim surrounding whitespace, and standardize case while preserving the original labels for audit. Distinct ticker symbols will remain separate; similarity in spelling is not sufficient evidence for reassignment.

Identifier normalization must precede grouping and duplicate detection so that differently formatted observations for the same ticker are evaluated together.

In [5]:
example_labels = pd.Series([
    "AAPL",
    '<span style="color: blue">AAPL</span>',
    " aapl ",
    "AMGN",
    "AMZN",
], dtype="string")

normalized_labels = (
    example_labels
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.strip()
    .str.upper()
)

display(pd.DataFrame({
    "original": example_labels,
    "normalized": normalized_labels,
}))

,original,normalized
0,AAPL,AAPL
1,"<span style=""color: blue"">AAPL</span>",AAPL
2,aapl,AAPL
3,AMGN,AMGN
4,AMZN,AMZN


## 3. Normalize ticker identifiers

**Purpose:** Ensure that presentation differences do not split a security’s observations into separate groups.

Remove the simple HTML tags observed in the delivery, trim surrounding whitespace, and standardize ticker case. A small example confirmed that these operations consolidate formatting variants while preserving distinct symbols.

Apply normalization to a working copy and retain each original ticker label in `ticker_raw`. This step changes identifiers only; it does not remove observations or repair market values.

In [6]:
working = raw.copy()
working["ticker_raw"] = working["ticker"]

working["ticker"] = (
    working["ticker_raw"]
    .astype("string")
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.strip()
    .str.upper()
)

In [7]:
print("Raw labels:", raw["ticker"].nunique())
print("Normalized tickers:", working["ticker"].nunique())

assert len(working) == len(raw), "Normalization changed the row count."
assert working["ticker_raw"].equals(raw["ticker"]), "Original labels changed."
assert working["ticker"].notna().all(), "Missing normalized ticker."
assert working["ticker"].ne("").all(), "Empty normalized ticker."

ticker_audit = (
    working[["ticker_raw", "ticker"]]
    .drop_duplicates()
    .sort_values(["ticker", "ticker_raw"])
)

display(ticker_audit.head(10))

Raw labels: 150
Normalized tickers: 30


,ticker_raw,ticker
102,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
1121,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
825,"<a target=""_blank""><span style=""font-weight: b...",AAPL
744,"<a target=""_blank""><span style=""font-weight: b...",AAPL
4,AAPL,AAPL
5005,"<a target=""_blank""><span style=""color: blue"">A...",AMGN
45,"<a target=""_blank""><span style=""color: red"">AM...",AMGN
368,"<a target=""_blank""><span style=""font-weight: b...",AMGN
1227,"<a target=""_blank""><span style=""font-weight: b...",AMGN
33,AMGN,AMGN


**Normalization validation:** The transformation preserved the number of observations and retained the original ticker labels in `ticker_raw`. Checks found no missing or empty normalized identifiers. The displayed audit rows confirm that HTML-wrapped and plain-text variants map to the same ticker.

No market values were modified and no observations were removed. Duplicate observations and conflicting ticker-date keys remain to be assessed.

Normalization reduced the number of distinct ticker labels from 150 to 30 while preserving all 397,711 observations. This reduction reflects consolidation of formatting variants, not removal of market-data rows.

In [8]:
# Compare the original data fields, using the now-normalized ticker.
observation_columns = raw.columns.tolist()

exact_duplicate_mask = working.duplicated(
    subset=observation_columns,
    keep="first",
)

print("Extra exact duplicate rows:", exact_duplicate_mask.sum())

Extra exact duplicate rows: 29460


### Exact duplicates

After ticker normalization, 29,460 rows were identified as additional copies of otherwise identical observations. Equality was assessed across the eight original data fields, using normalized ticker values and excluding the audit field `ticker_raw`.

Retain one copy of each exact observation. Preserve the original delivery and the ticker-normalization audit separately. The expected remaining row count is 368,251.

This check does not establish uniqueness of `(ticker, ts)`: records sharing that key may still contain conflicting market values.

In [9]:
deduplicated = (
    working.loc[~exact_duplicate_mask]
    .copy()
    .reset_index(drop=True)
)

assert len(deduplicated) == len(working) - exact_duplicate_mask.sum()

conflicting_key_mask = deduplicated.duplicated(
    subset=["ticker", "ts"],
    keep=False,
)

print("Rows after exact deduplication:", len(deduplicated))
print("Rows sharing conflicting keys:", conflicting_key_mask.sum())

display(
    deduplicated.loc[conflicting_key_mask]
    .sort_values(["ticker", "ts"])
    .head(10)
)

Rows after exact deduplication: 368251
Rows sharing conflicting keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,ticker_raw


**Deduplication validation:** Removing 29,460 extra exact copies reduced the dataset from 397,711 to 368,251 observations. No repeated `(ticker, ts)` keys remained after this step, so no conflicting-key resolution was required.

The empty conflict preview is consistent with the zero count. Key uniqueness does not establish valid prices or complete trading-date coverage; those checks remain outstanding.

In [10]:
if conflicting_key_mask.any():
    display(
        deduplicated.loc[conflicting_key_mask]
        .sort_values(["ticker", "ts"])
        .head(10)
    )
else:
    print("No conflicting ticker-date keys found.")

No conflicting ticker-date keys found.


## 4. Assess missing values

**Purpose:** Measure missing values after identifier normalization and exact deduplication.

Recalculate missing counts because duplicate observations contributed to the original delivery’s counts. Report both the number and proportion of missing values in each original data column, using the deduplicated row count as the denominator.

This check identifies missing cells in existing observations. Missing trading-date rows require a separate comparison with an expected calendar.

In [11]:
def summarize_missing_values(frame):
    """Return missing-cell counts and fractions for each supplied column."""
    return pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_fraction": frame.isna().mean(),
    })

In [12]:
example = pd.DataFrame({
    "close": [100.0, None, 102.0],
    "volume": [10, 20, 30],
})

example_summary = summarize_missing_values(example)

assert example_summary.loc["close", "missing_count"] == 1
assert example_summary.loc["close", "missing_fraction"] == 1 / 3
assert example_summary.loc["volume", "missing_count"] == 0

print("Missing-cell detector checks passed.")

Missing-cell detector checks passed.


In [13]:
missing_summary = summarize_missing_values(
    deduplicated[observation_columns]
)

# Format a separate display copy; preserve numeric values for analysis.
missing_display = missing_summary.copy()
missing_display["missing_fraction"] = (
    missing_display["missing_fraction"].map("{:.2%}".format)
)

display(missing_display)

,missing_count,missing_fraction
ts,0,0.00%
open,0,0.00%
high,0,0.00%
low,0,0.00%
close,0,0.00%
adj_close,18374,4.99%
volume,0,0.00%
ticker,0,0.00%


**Missing-value findings:** After exact deduplication, `adj_close` contains 18,374 missing values, representing approximately 4.99% of retained observations. All other original columns have zero missing cells.

The reduction from the original missing count reflects removal of duplicate observations, not imputation. Adjusted-close repair remains outstanding and will require a consistent adjustment convention.

These results do not establish valid price values or complete trading-date coverage. Nonpositive or infinite values are not necessarily missing, and absent observations cannot be detected by examining existing cells alone.

## 5. Detect missing trading dates

**Purpose:** Identify dates present in the reference history for a ticker but absent from the deduplicated homework data.

Use each ticker’s dates in the supplied `ohlc` table as its expected calendar, as permitted by the assignment. Compare histories separately by ticker rather than imposing one common date range on all securities.

This check assesses completeness relative to the supplied reference, not independently verified exchange coverage. Dates absent from both tables will not be detected. Report missing dates without creating observations or imputing prices.

Before applying the detector to the full dataset, verify it on a small example containing one deliberately removed date.

In [14]:
expected_dates = pd.to_datetime([
    "2024-01-02",
    "2024-01-03",
    "2024-01-04",
    "2024-01-05",
])

# Deliberately omit the second expected date.
observed_dates = expected_dates[[0, 2, 3]]

missing_dates = expected_dates.difference(observed_dates)

print("Missing dates:", missing_dates.strftime("%Y-%m-%d").tolist())

Missing dates: ['2024-01-03']


In [15]:
def find_missing_dates(observed_dates, expected_dates):
    """Return sorted expected dates absent from one ticker's observations."""
    observed = pd.DatetimeIndex(observed_dates).dropna().unique()
    expected = pd.DatetimeIndex(expected_dates).dropna().unique()

    return expected.difference(observed).sort_values()

In [16]:
detected = find_missing_dates(observed_dates, expected_dates)
complete = find_missing_dates(expected_dates, expected_dates)

assert detected.equals(pd.DatetimeIndex(["2024-01-03"]))
assert complete.empty

print("Missing-date detector checks passed.")

Missing-date detector checks passed.


**Detector verification:** The controlled example identified January 3, 2024 as the sole missing expected date. A second check using identical observed and expected dates returned no gaps. Both assertions passed.

The detector compares supplied date sets; it does not infer exchange sessions or create missing observations.

In [17]:
with closing(sqlite3.connect(source_uri, uri=True)) as conn:
    reference_keys = pd.read_sql_query(
        "SELECT ticker, ts FROM ohlc",
        conn,
        parse_dates=["ts"],
    )

display(reference_keys.head())
print("Reference rows:", len(reference_keys))
print("Distinct reference labels:", reference_keys["ticker"].nunique())

,ticker,ts
0,JNJ,2024-10-14
1,JNJ,2024-10-11
2,JNJ,2024-10-10
3,JNJ,2024-09-30
4,JNJ,2024-09-27


Reference rows: 368251
Distinct reference labels: 30


### Reference-calendar inspection

Loaded only `ticker` and `ts` from the supplied `ohlc` table through a read-only connection. The reference contains 368,251 rows and 30 distinct raw ticker labels.

These counts match those of the deduplicated homework dataset, but count agreement alone does not establish that the ticker-date keys match. The query did not specify an ordering, so the preview is not evidence of chronological continuity.

Before comparing dates, inspect and normalize reference ticker labels using the same convention as the homework data. Check for missing identifiers or dates and repeated reference keys.

In [18]:
reference_calendar = reference_keys.copy()

reference_calendar["ticker"] = (
    reference_calendar["ticker"]
    .astype("string")
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.strip()
    .str.upper()
)

reference_label_audit = pd.DataFrame({
    "original": reference_keys["ticker"],
    "normalized": reference_calendar["ticker"],
}).drop_duplicates()

display(reference_label_audit.sort_values("normalized"))

assert reference_calendar["ticker"].notna().all()
assert reference_calendar["ticker"].ne("").all()
assert reference_calendar["ts"].notna().all()

print(
    "Extra reference ticker-date duplicates:",
    reference_calendar.duplicated(["ticker", "ts"]).sum(),
)

homework_tickers = set(deduplicated["ticker"])
reference_tickers = set(reference_calendar["ticker"])

print("Homework tickers without a reference:", sorted(
    homework_tickers - reference_tickers
))
print("Reference tickers absent from homework:", sorted(
    reference_tickers - homework_tickers
))

,original,normalized
9234,AAPL,AAPL
13715,AMGN,AMGN
166161,AMZN,AMZN
13749,AXP,AXP
284301,BA,BA
31167,CAT,CAT
25,CRM,CRM
211267,CSCO,CSCO
87160,CVX,CVX
97222,DIS,DIS


Extra reference ticker-date duplicates: 0


Homework tickers without a reference: []
Reference tickers absent from homework: []


**Reference-key validation:** All 30 reference ticker labels were unchanged by normalization. Checks found no missing or empty reference tickers, missing reference dates, or duplicate `(ticker, ts)` keys. The reference and homework ticker sets match.

These checks establish that each homework ticker has an available reference calendar. Date coverage within each ticker remains to be compared.

In [19]:
coverage_rows = []
missing_date_rows = []

# Use each ticker's supplied history rather than one shared calendar.
for ticker, reference_group in reference_calendar.groupby("ticker"):
    expected = reference_group["ts"]

    # Restrict observed dates to the same security before comparing.
    observed = deduplicated.loc[
        deduplicated["ticker"].eq(ticker), "ts"
    ]

    gaps = find_missing_dates(observed, expected)

    coverage_rows.append({
        "ticker": ticker,
        "start_date": expected.min(),
        "end_date": expected.max(),
        "expected_days": expected.nunique(),
        "missing_days": len(gaps),
    })

    for date in gaps:
        missing_date_rows.append({
            "ticker": ticker,
            "issue_date": date,
        })

coverage_summary = pd.DataFrame(coverage_rows)

# Explicit columns keep the structure usable even when no gaps exist.
missing_date_details = pd.DataFrame(
    missing_date_rows,
    columns=["ticker", "issue_date"],
)

display(coverage_summary)
print("Total missing ticker-date observations:",
      coverage_summary["missing_days"].sum())

,ticker,start_date,end_date,expected_days,missing_days
0,AAPL,1980-12-12,2024-10-18,11050,0
1,AMGN,1983-06-17,2024-10-18,10421,0
2,AMZN,1997-05-15,2024-12-06,6937,0
3,AXP,1972-04-03,2024-10-18,13252,0
4,BA,1962-01-02,2024-10-18,15784,0
5,CAT,1962-01-02,2024-10-18,15808,0
6,CRM,2004-06-23,2024-10-18,5117,0
7,CSCO,1990-02-16,2024-10-18,8734,0
8,CVX,1962-01-02,2024-12-06,15843,0
9,DIS,1962-01-02,2024-12-06,15867,0


Total missing ticker-date observations: 0


In [20]:
for row in coverage_summary.itertuples(index=False):
    print(
        f"Ticker {row.ticker} is missing {row.missing_days} trading days "
        f"between {row.start_date:%Y-%m-%d} and {row.end_date:%Y-%m-%d} "
        f"relative to the supplied reference."
    )

Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18 relative to the supplied reference.
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18 relative to the supplied reference.
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06 relative to the supplied reference.
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18 relative to the supplied reference.
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18 relative to the supplied reference.
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18 relative to the supplied reference.
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18 relative to the supplied reference.
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18 relative to the supplied reference.
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06 relative to the supplied reference.
Ticker DIS is missing 0 trading da

**Trading-date coverage findings:** Across all 30 tickers, no expected ticker-date observations were missing from the normalized, deduplicated homework data relative to each ticker’s supplied `ohlc` history.

No rows or prices were created to fill calendar gaps. This result is conditional on the supplied reference: dates absent from both datasets would remain undetected.

The detector previously passed a controlled test with one deliberately removed date, demonstrating that a zero-gap result on this delivery does not arise merely from testing a case with no missing dates.

## 6. Validate price values and OHLC relationships

**Purpose:** Identify invalid equity prices and internally inconsistent daily price ranges before selecting any repairs.

Check all raw OHLC fields and adjusted close for missing, non-finite, or nonpositive values. Check raw open and close against the raw daily low and high, and explicitly identify inverted ranges where low exceeds high.

Adjusted close is assessed separately from raw OHLC bounds because the adjustment convention may place it on a different scale.

These checks identify violations, not their causes. An inconsistent row does not by itself establish which field is incorrect. Retain the observations for diagnosis; no repairs are performed in this step.

In [21]:
import numpy as np

example_prices = pd.Series(
    [100.0, 0.0, -5.0, np.inf, np.nan],
    name="price",
)

price_checks = pd.DataFrame({
    "price": example_prices,
    "missing": example_prices.isna(),
    "nonfinite": ~np.isfinite(example_prices),
    "nonpositive": example_prices.le(0),
})

display(price_checks)

,price,missing,nonfinite,nonpositive
0,100.0,False,False,False
1,0.0,False,False,True
2,-5.0,False,False,True
3,inf,False,True,False
4,NaN,True,True,False


In [22]:
def detect_invalid_prices(frame):
    """Return Boolean issue flags for raw and adjusted price values."""
    price_columns = ["open", "high", "low", "close", "adj_close"]
    prices = frame[price_columns]

    return pd.DataFrame({
        "missing_price": prices.isna().any(axis=1),
        "nonfinite_price": (~np.isfinite(prices)).any(axis=1),
        "nonpositive_price": prices.le(0).any(axis=1),
    }, index=frame.index)

In [23]:
# Start with five valid observations, then inject specific faults.
test_prices = pd.DataFrame({
    column: [100.0] * 5
    for column in ["open", "high", "low", "close", "adj_close"]
})

test_prices.loc[1, "open"] = 0.0
test_prices.loc[2, "close"] = -5.0
test_prices.loc[3, "high"] = np.inf
test_prices.loc[4, "adj_close"] = np.nan

test_flags = detect_invalid_prices(test_prices)

assert test_flags["missing_price"].tolist() == [
    False, False, False, False, True
]
assert test_flags["nonfinite_price"].tolist() == [
    False, False, False, True, True
]
assert test_flags["nonpositive_price"].tolist() == [
    False, True, True, False, False
]

print("Price-validity detector checks passed.")

Price-validity detector checks passed.


**Price-value detector verification:** Controlled cases confirmed that the detector distinguishes valid prices, zero or negative prices, infinity, and missing adjusted close. Missing values also trigger the non-finite flag by design. All expected Boolean flags matched.

The detector checks value validity only; OHLC relationships are assessed separately. Flags identify observations requiring review without modifying their values.

In [24]:
price_value_flags = detect_invalid_prices(deduplicated)

display(
    price_value_flags.sum().rename("flagged_observations").to_frame()
)

print(
    "Observations with at least one price-value issue:",
    price_value_flags.any(axis=1).sum(),
)

,flagged_observations
missing_price,18374
nonfinite_price,18374
nonpositive_price,5


Observations with at least one price-value issue: 18379


In [25]:
nonpositive_rows = deduplicated.loc[
    price_value_flags["nonpositive_price"],
    ["ticker", "ts", "open", "high", "low", "close", "adj_close"],
].sort_values(["ticker", "ts"])

display(nonpositive_rows)

,ticker,ts,open,high,low,close,adj_close
271962,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750
327908,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012
215385,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090
339808,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447
97792,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300


### Price-value findings

The detector flagged 18,374 observations with missing prices, 18,374 with non-finite prices, and 5 with nonpositive prices. Missing prices also trigger the non-finite flag, so these counts are not additive.

A total of 18,379 observations have at least one price-value issue. The five nonpositive-price observations are separate from the missing-price observations; each has an opening price of zero.

The affected ticker-date pairs are CVX, JNJ, MRK, and PG on January 2, 1962, and MCD on July 5, 1966. Each zero opening price also falls below its reported daily low.

The cause of these values has not been established. No repair has been performed. Historical coverage and supporting evidence will be examined before choosing a repair or quarantine decision.

In [26]:
def detect_ohlc_issues(frame):
    """Flag violations of raw daily OHLC bounds; exclude adjusted close."""
    return pd.DataFrame({
        "open_below_low": frame["open"].lt(frame["low"]),
        "open_above_high": frame["open"].gt(frame["high"]),
        "close_below_low": frame["close"].lt(frame["low"]),
        "close_above_high": frame["close"].gt(frame["high"]),
        "inverted_range": frame["low"].gt(frame["high"]),
    }, index=frame.index)

In [27]:
ohlc_example = pd.DataFrame({
    "open": [100.0],
    "high": [105.0],
    "low": [95.0],
    "close": [108.0],
})

example_flags = detect_ohlc_issues(ohlc_example)

assert example_flags.loc[0, "close_above_high"]
assert example_flags.loc[0].sum() == 1

display(example_flags)
print("Close-above-high example passed.")

,open_below_low,open_above_high,close_below_low,close_above_high,inverted_range
0,False,False,False,True,False


Close-above-high example passed.


### OHLC detector verification

Test valid observations, prices equal to the daily bounds, each open/close bound violation, and an inverted high-low range. Compare every returned flag against an explicitly specified expected result.

An inverted range may trigger additional bound violations; multiple flags on one observation are permitted.

In [28]:
ohlc_cases = pd.DataFrame(
    [
        [100, 105, 95, 102],  # Valid interior prices.
        [95, 105, 95, 105],   # Equality at the bounds is valid.
        [94, 105, 95, 100],   # Open below low.
        [106, 105, 95, 100],  # Open above high.
        [100, 105, 95, 94],   # Close below low.
        [100, 105, 95, 106],  # Close above high.
        [100, 95, 105, 100],  # Inverted range: all five conditions apply.
    ],
    columns=["open", "high", "low", "close"],
    index=[
        "valid", "boundary", "open_low", "open_high",
        "close_low", "close_high", "inverted",
    ],
)

expected_flags = pd.DataFrame(
    [
        [False, False, False, False, False],
        [False, False, False, False, False],
        [True,  False, False, False, False],
        [False, True,  False, False, False],
        [False, False, True,  False, False],
        [False, False, False, True,  False],
        [True,  True,  True,  True,  True],
    ],
    columns=[
        "open_below_low", "open_above_high",
        "close_below_low", "close_above_high", "inverted_range",
    ],
    index=ohlc_cases.index,
)

pd.testing.assert_frame_equal(
    detect_ohlc_issues(ohlc_cases),
    expected_flags,
)

print("All seven OHLC test cases passed.")

All seven OHLC test cases passed.


In [29]:
ohlc_flags = detect_ohlc_issues(deduplicated)

display(
    ohlc_flags.sum().rename("flagged_observations").to_frame()
)

print(
    "Observations with at least one OHLC issue:",
    ohlc_flags.any(axis=1).sum(),
)

,flagged_observations
open_below_low,5
open_above_high,0
close_below_low,0
close_above_high,0
inverted_range,0


Observations with at least one OHLC issue: 5


In [30]:
assert ohlc_flags.any(axis=1).equals(
    price_value_flags["nonpositive_price"]
)

print("OHLC violations match the five nonpositive-price observations.")

OHLC violations match the five nonpositive-price observations.


**OHLC verification and findings:** All seven controlled test cases passed, including valid boundary equality, each individual bound violation, and an inverted range.

In the homework data, five observations have opening prices below their daily lows. No openings above the high, closes outside the daily range, or inverted ranges were detected. A row-level comparison confirmed that the five affected observations are exactly the previously identified zero-opening-price observations.

These observations carry both nonpositive-price and OHLC-bound flags. They represent five affected rows, not ten. No repairs have been performed.

## 7. Validate trading volume

**Purpose:** Distinguish missing, invalid, and reported zero-volume observations before applying repairs.

Missing volume represents an unknown quantity. Negative or infinite volume is invalid. Reported zero volume is a review flag rather than conclusive evidence of an error or stale prices.

The assignment requires replacing missing volume with zero during cleaning while retaining a repair flag. This preserves the distinction between reported and imputed zeros for subsequent analysis. No volume repairs are performed in this diagnostic step.

In [31]:
def detect_volume_issues(frame):
    """Flag missing, infinite, negative, and reported zero volume."""
    volume = frame["volume"]

    return pd.DataFrame({
        "missing_volume": volume.isna(),
        # Separate infinity from missingness for clearer issue reporting.
        "infinite_volume": np.isinf(volume),
        "negative_volume": volume.lt(0),
        "zero_volume": volume.eq(0),
    }, index=frame.index)

In [32]:
volume_example = pd.DataFrame({
    "volume": [100.0, np.nan, np.inf, -5.0, 0.0],
})

expected_volume_flags = pd.DataFrame({
    "missing_volume":  [False, True,  False, False, False],
    "infinite_volume": [False, False, True,  False, False],
    "negative_volume": [False, False, False, True,  False],
    "zero_volume":     [False, False, False, False, True],
})

pd.testing.assert_frame_equal(
    detect_volume_issues(volume_example),
    expected_volume_flags,
)

print("Volume detector example checks passed.")

Volume detector example checks passed.


In [33]:
volume_flags = detect_volume_issues(deduplicated)

display(
    volume_flags.sum().rename("flagged_observations").to_frame()
)

print(
    "Observations with at least one volume flag:",
    volume_flags.any(axis=1).sum(),
)

,flagged_observations
missing_volume,0
infinite_volume,0
negative_volume,0
zero_volume,75


Observations with at least one volume flag: 75


**Volume verification and findings:** The detector passed the controlled examples for positive, missing, infinite, negative, and zero volume.

In the deduplicated homework data, no missing, infinite, or negative volume values were detected. There are 75 reported zero-volume observations. These remain review flags; their cause has not been established, and no volume values have been changed.

Missing-volume repair behavior must still be demonstrated on a controlled example because the actual delivery contains no missing volume.

## 8. Screen for extreme daily returns

**Purpose:** Identify unusually large positive or negative raw-close returns for investigation.

Calculate simple returns within each ticker after chronological sorting. Flag returns more than three sample standard deviations from that ticker’s mean return, using `ddof=1`.

This retrospective screen uses the supplied history and is not a deletion rule or a trading signal. Large observations can influence both the estimated mean and standard deviation. Flagged returns require supporting evidence before any repair decision.

The first observation for each ticker has no preceding close and therefore no defined return. Missing or invalid closes must not be silently filled for this calculation.

In [34]:
return_review = deduplicated.sort_values(
    ["ticker", "ts"]
).copy()

# Invalid closes cannot serve as either endpoint of a usable return.
valid_close = return_review["close"].where(
    np.isfinite(return_review["close"])
    & return_review["close"].gt(0)
)

return_review["simple_return"] = (
    valid_close
    .groupby(return_review["ticker"])
    .pct_change(fill_method=None)
)

display(
    return_review.loc[
        return_review["ticker"].eq("AAPL"),
        ["ticker", "ts", "close", "simple_return"],
    ].head()
)

,ticker,ts,close,simple_return
71438,AAPL,1980-12-12,0.12835,NaN
100357,AAPL,1980-12-15,0.12165,-0.052201
278091,AAPL,1980-12-16,0.11272,-0.073407
269953,AAPL,1980-12-17,0.11551,0.024752
187275,AAPL,1980-12-18,0.11886,0.029002


**Initial return check:** AAPL’s first observation has an undefined return because no preceding close is available. The next observation’s simple return agrees with the direct calculation `0.12165 / 0.12835 - 1`, approximately −5.2201%.

This spot check supports the calculation for the displayed observations; broader grouped-return and threshold tests remain outstanding.

In [35]:
grouped_returns = return_review.groupby("ticker")["simple_return"]

return_review["mean_return"] = grouped_returns.transform("mean")
return_review["std_return"] = grouped_returns.transform(
    lambda values: values.std(ddof=1)
)

# A statistical flag requires a finite return and a positive, finite std.
screen_defined = (
    np.isfinite(return_review["simple_return"])
    & np.isfinite(return_review["mean_return"])
    & np.isfinite(return_review["std_return"])
    & return_review["std_return"].gt(0)
)

return_review["extreme_return"] = (
    screen_defined
    & (
        return_review["simple_return"]
        - return_review["mean_return"]
    ).abs().gt(3 * return_review["std_return"])
)

extreme_counts = (
    return_review.groupby("ticker")["extreme_return"]
    .sum()
    .rename("flagged_returns")
)

display(extreme_counts.to_frame())
print("Total extreme-return flags:", extreme_counts.sum())

,flagged_returns
ticker,
AAPL,143
AMGN,172
AMZN,147
AXP,199
BA,210
CAT,221
CRM,82
CSCO,148
CVX,178


Total extreme-return flags: 5056


**Initial statistical-screen result:** The within-ticker, two-sided three-standard-deviation screen flagged 5,056 simple raw-close returns.

These flags identify observations for investigation, not confirmed data errors. The screen uses full-history sample statistics and is retrospective. No flagged observations have been removed or repaired. Controlled tests of both tails and degenerate cases follow.

In [36]:
def flag_extreme_returns(returns):
    """Flag finite returns more than three sample stds from their mean."""
    observed = returns.loc[np.isfinite(returns)]
    flags = pd.Series(False, index=returns.index, dtype=bool)

    if len(observed) < 2:
        return flags

    mean_return = observed.mean()
    std_return = observed.std(ddof=1)

    if not np.isfinite(std_return) or std_return <= 0:
        return flags

    flags.loc[observed.index] = (
        (observed - mean_return).abs() > 3 * std_return
    )
    return flags

In [37]:
# Enough ordinary observations keep the two extremes beyond the threshold.
two_tails = pd.Series([0.0] * 30 + [0.50, -0.50, np.nan])
flags = flag_extreme_returns(two_tails)

assert flags.iloc[30]
assert flags.iloc[31]
assert flags.sum() == 2
assert not flags.iloc[32]

assert not flag_extreme_returns(pd.Series([0.0] * 5)).any()
assert not flag_extreme_returns(pd.Series([0.02, np.nan])).any()
assert not flag_extreme_returns(
    pd.Series([np.nan, np.inf, -np.inf])
).any()

print("Extreme-return detector checks passed.")

Extreme-return detector checks passed.


In [38]:
function_flags = (
    return_review.groupby("ticker")["simple_return"]
    .transform(flag_extreme_returns)
)

pd.testing.assert_series_equal(
    function_flags,
    return_review["extreme_return"],
    check_names=False,
)

print("Function matches the original screen.")

Function matches the original screen.


### Extreme-return frequency by ticker

Summarize flagged returns as both counts and fractions of each ticker’s finite returns. Using a ticker-specific denominator accounts for differences in available history.

This summary guides investigation; it is not a ranking of data quality. Genuine return distributions may differ across securities and need not follow a normal distribution.

In [39]:
return_screen_summary = (
    return_review
    .assign(
        finite_return=np.isfinite(return_review["simple_return"])
    )
    .groupby("ticker")
    .agg(
        usable_returns=("finite_return", "sum"),
        flagged_returns=("extreme_return", "sum"),
    )
)

# No usable returns means the flag rate is undefined, rather than zero.
return_screen_summary["flagged_fraction"] = (
    return_screen_summary["flagged_returns"]
    / return_screen_summary["usable_returns"].replace(0, np.nan)
)

return_screen_summary = return_screen_summary.sort_values(
    "flagged_fraction", ascending=False
)

# Preserve numeric fractions in the analytical table.
screen_display = return_screen_summary.copy()
screen_display["flagged_fraction"] = (
    screen_display["flagged_fraction"].map(
        lambda value: "N/A" if pd.isna(value) else f"{value:.2%}"
    )
)

display(screen_display)

,usable_returns,flagged_returns,flagged_fraction
ticker,,,
AMZN,6936,147,2.12%
CSCO,8733,148,1.69%
AMGN,10420,172,1.65%
UNH,10082,163,1.62%
V,4175,67,1.60%
CRM,5116,82,1.60%
NKE,11097,177,1.60%
WMT,13251,209,1.58%
AXP,13251,199,1.50%


**Flag-frequency findings:** AMZN has the highest flagged fraction at approximately 2.12% (147 of 6,936 usable returns), followed by CSCO, AMGN, UNH, and V.

These differences guide investigation but do not establish differences in data quality. The screen depends on each security’s return distribution and supplied history.

In [40]:
amzn_history = (
    return_review.loc[return_review["ticker"].eq("AMZN")]
    .sort_values("ts")
    .reset_index(drop=True)
)

flagged_returns = amzn_history.loc[
    amzn_history["extreme_return"], "simple_return"
]

if flagged_returns.empty:
    raise ValueError("No flagged AMZN returns to inspect.")

event_position = flagged_returns.abs().idxmax()

# Include up to two observations before and after the selected event.
event_window = amzn_history.iloc[
    max(0, event_position - 2):event_position + 3
]

display(event_window[[
    "ticker", "ts", "open", "high", "low",
    "close", "adj_close", "volume", "simple_return",
    "extreme_return",
]])

,ticker,ts,open,high,low,close,adj_close,volume,simple_return,extreme_return
1137,AMZN,2001-11-21,0.4370,0.4485,0.402,0.4400,0.4400,179932000,0.023256,False
1138,AMZN,2001-11-23,0.4450,0.4570,0.428,0.4540,0.4540,81456000,0.031818,False
1139,AMZN,2001-11-26,0.4915,0.6110,0.490,0.6105,0.6105,1013784000,0.344714,True
1140,AMZN,2001-11-27,0.6025,0.6125,0.561,0.5740,0.5740,686176000,-0.059787,False
1141,AMZN,2001-11-28,0.5620,0.6200,0.559,0.5795,0.5795,970324000,0.009582,False


**Extreme-return case review — AMZN, November 26, 2001:** The raw-close simple return is approximately +34.47%. The reported close lies within the daily high-low range, and adjusted close equals raw close throughout the displayed five-observation window.

The statistical flag alone does not justify repair. Retain the observation with its review flag unless additional evidence establishes an error. No specific corporate event or news explanation has been verified.

## 9. Compare prices with the supplied reference

Use matching normalized `(ticker, ts)` keys to compare homework prices with the supplied `ohlc` values. Validate reference-key uniqueness before joining to prevent accidental row multiplication.

Reference values support investigation and date-specific adjustment factors. They will not replace the homework dataset wholesale. Agreement with the reference does not establish independent market accuracy.

In [41]:
with closing(sqlite3.connect(source_uri, uri=True)) as conn:
    reference_prices = pd.read_sql_query(
        """
        SELECT ticker, ts, open, high, low, close, adj_close
        FROM ohlc
        """,
        conn,
        parse_dates=["ts"],
    )

reference_prices["ticker"] = (
    reference_prices["ticker"].astype("string")
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.strip()
    .str.upper()
)

assert not reference_prices.duplicated(["ticker", "ts"]).any()

price_columns = ["open", "high", "low", "close", "adj_close"]

comparison = deduplicated.merge(
    reference_prices,
    on=["ticker", "ts"],
    how="left",
    suffixes=("", "_ref"),
    validate="one_to_one",
    indicator=True,
)

assert len(comparison) == len(deduplicated)
assert comparison["_merge"].eq("both").all()

comparison_rows = []

for column in price_columns:
    homework_values = comparison[column]
    reference_values = comparison[f"{column}_ref"]

    both_finite = (
        np.isfinite(homework_values)
        & np.isfinite(reference_values)
    )

    # Compare numerical values with an explicit floating-point tolerance.
    differs = both_finite & ~np.isclose(
        homework_values,
        reference_values,
        rtol=1e-8,
        atol=1e-10,
    )

    comparison_rows.append({
        "column": column,
        "homework_missing": homework_values.isna().sum(),
        "reference_missing": reference_values.isna().sum(),
        "finite_value_disagreements": differs.sum(),
    })

display(pd.DataFrame(comparison_rows).set_index("column"))

display(
    comparison.loc[
        comparison["open"].eq(0),
        ["ticker", "ts", "open", "open_ref", "close", "close_ref"],
    ]
)

,homework_missing,reference_missing,finite_value_disagreements
column,,,
open,0,0,0
high,0,0,0
low,0,0,0
close,0,0,0
adj_close,18374,0,10417


,ticker,ts,open,open_ref,close,close_ref
97792,PG,1962-01-02,0.0,0.0,1.43000,1.43000
215385,MCD,1966-07-05,0.0,0.0,0.26955,0.26955
271962,CVX,1962-01-02,0.0,0.0,3.30000,3.30000
327908,JNJ,1962-01-02,0.0,0.0,0.22338,0.22338
339808,MRK,1962-01-02,0.0,0.0,0.39537,0.39537


### Reference comparison findings

All finite raw OHLC values match the supplied reference within the stated numerical tolerance. The five zero opening prices also appear in the reference, so reference substitution would not resolve those violations.

Adjusted close contains 18,374 missing values and 10,417 finite disagreements with the reference. These discrepancies require investigation before reconstruction using the assignment-permitted date-specific reference adjustment factors.

Reference agreement supports consistency with the supplied source, not independent market verification. No prices have been changed.

In [42]:
reference_valid = (
    np.isfinite(comparison["close_ref"])
    & comparison["close_ref"].gt(0)
    & np.isfinite(comparison["adj_close_ref"])
    & comparison["adj_close_ref"].gt(0)
)

comparison["reference_factor"] = (
    comparison["adj_close_ref"] / comparison["close_ref"]
).where(reference_valid)

adjusted_both_finite = (
    np.isfinite(comparison["adj_close"])
    & np.isfinite(comparison["adj_close_ref"])
)

adjusted_disagrees = adjusted_both_finite & ~np.isclose(
    comparison["adj_close"],
    comparison["adj_close_ref"],
    rtol=1e-8,
    atol=1e-10,
)

adjusted_examples = comparison.loc[
    adjusted_disagrees,
    ["ticker", "ts", "close", "adj_close", "adj_close_ref"],
].copy()

adjusted_examples["homework_to_reference_ratio"] = (
    adjusted_examples["adj_close"]
    / adjusted_examples["adj_close_ref"]
)

print("Rows without a valid reference factor:", (~reference_valid).sum())

display(
    adjusted_examples.sort_values(
        "homework_to_reference_ratio", ascending=False
    ).head(10)
)

display(adjusted_examples["homework_to_reference_ratio"].describe())

Rows without a valid reference factor: 0


,ticker,ts,close,adj_close,adj_close_ref,homework_to_reference_ratio
366613,NKE,2008-06-24,16.49000,319.899187,13.33000,23.998439
200663,TRV,1984-09-26,6.47000,47.749865,1.99000,23.994907
341976,NKE,2007-02-01,12.58000,238.987226,9.96000,23.994701
126183,AXP,1999-08-27,42.59000,726.546776,30.28000,23.994279
351075,MRK,1989-07-24,11.78000,87.088464,3.63000,23.991312
363114,VZ,2008-05-19,36.53000,377.838300,15.75000,23.989733
153323,JPM,1990-09-17,6.50000,55.166880,2.30000,23.985600
147813,NVDA,2003-07-23,0.17975,3.953666,0.16487,23.980503
161168,KO,1990-02-23,4.25000,44.603505,1.86000,23.980379
185751,NKE,2020-08-31,111.89000,2552.228725,106.43000,23.980351


count    10417.000000
mean        14.081361
std          5.761729
min          4.002156
25%          9.185794
50%         14.064218
75%         19.060007
max         23.998439
Name: homework_to_reference_ratio, dtype: float64

In [43]:
first_dates = comparison.groupby("ticker")["ts"].transform("min")

zero_open_review = comparison.loc[
    comparison["open"].eq(0),
    ["ticker", "ts", "open", "open_ref"],
].copy()

zero_open_review["is_first_observation"] = (
    comparison.loc[zero_open_review.index, "ts"]
    .eq(first_dates.loc[zero_open_review.index])
)

display(zero_open_review)

,ticker,ts,open,open_ref,is_first_observation
97792,PG,1962-01-02,0.0,0.0,True
215385,MCD,1966-07-05,0.0,0.0,True
271962,CVX,1962-01-02,0.0,0.0,True
327908,JNJ,1962-01-02,0.0,0.0,True
339808,MRK,1962-01-02,0.0,0.0,True


### Evidence supporting the proposed cleaning policy

The 10,417 finite adjusted-close disagreements have homework-to-reference ratios ranging from approximately 4.002 to 23.998. These discrepancies are substantial relative to the stated comparison tolerance. Their underlying cause has not been established.

All five zero opening prices occur at the beginning of their respective supplied ticker histories and also appear in the reference. Forward filling cannot resolve these leading observations. Quarantine is proposed rather than inventing an opening price.

For missing or inconsistent adjusted close, the proposed repair uses the supplied reference’s validated, date-specific adjustment factor multiplied by validated raw close. Apply this reconstruction only to affected observations, preserve original values and repair flags, and quarantine observations whose adjustment factor cannot be validated.

## 10. Consolidate the pre-repair Data Quality Report

Combine detected issues into one table with columns `ticker`, `issue_type`, `issue_date`, `description`, and `severity`. Allow multiple issues per observation; issue-row counts therefore differ from affected-observation counts.

Severity conventions:

- **High:** missing, non-finite, or nonpositive prices; invalid OHLC bounds; adjusted-price inconsistencies; and missing, infinite, or negative volume. These compromise required values or validation rules.
- **Medium:** extreme returns, reported zero volume, and absent reference-calendar dates. These require review but do not independently establish incorrect prices.

Exact duplicate removals and ticker normalization are documented separately in the preprocessing audit. Preserve this report before making repairs.

The accepted cleaning policy is to quarantine unresolved leading zero-open observations and reconstruct affected adjusted closes using validated, same-date reference adjustment factors. Original values and repair flags will be retained.

In [44]:
def issue_rows(frame, mask, issue_type, description, severity):
    """Convert an aligned Boolean issue mask into quality-report rows."""
    rows = frame.loc[mask, ["ticker", "ts"]].copy()
    rows = rows.rename(columns={"ts": "issue_date"})
    rows["issue_type"] = issue_type
    rows["description"] = description
    rows["severity"] = severity

    return rows[
        ["ticker", "issue_type", "issue_date", "description", "severity"]
    ]


issue_parts = []

# Existing detectors retain the deduplicated dataset's row index.
for issue_type in price_value_flags.columns:
    issue_parts.append(issue_rows(
        deduplicated,
        price_value_flags[issue_type],
        issue_type,
        f"At least one raw or adjusted price triggers {issue_type}.",
        "high",
    ))

for issue_type in ohlc_flags.columns:
    issue_parts.append(issue_rows(
        deduplicated,
        ohlc_flags[issue_type],
        issue_type,
        f"Raw OHLC relationship violated: {issue_type}.",
        "high",
    ))

for issue_type in volume_flags.columns:
    issue_parts.append(issue_rows(
        deduplicated,
        volume_flags[issue_type],
        issue_type,
        f"Reported volume triggers {issue_type}.",
        "medium" if issue_type == "zero_volume" else "high",
    ))

# Use each mask with the DataFrame from which it was calculated.
issue_parts.append(issue_rows(
    return_review,
    return_review["extreme_return"],
    "extreme_return",
    "Simple raw-close return exceeds the within-ticker 3-sigma threshold.",
    "medium",
))

issue_parts.append(issue_rows(
    comparison,
    adjusted_disagrees,
    "adjusted_price_inconsistency",
    "Adjusted close differs from the trusted date-specific reference convention.",
    "high",
))

gap_issues = missing_date_details.copy()
gap_issues["issue_type"] = "missing_trading_date"
gap_issues["description"] = "Expected date absent relative to supplied ohlc history."
gap_issues["severity"] = "medium"
issue_parts.append(gap_issues)

report_columns = [
    "ticker", "issue_type", "issue_date", "description", "severity"
]

# Exclude empty issue tables to avoid the pandas concatenation warning.
nonempty_parts = [part for part in issue_parts if not part.empty]

if nonempty_parts:
    quality_report_before = (
        pd.concat(nonempty_parts, ignore_index=True)
        [report_columns]
        .sort_values(["ticker", "issue_date", "issue_type"])
        .reset_index(drop=True)
    )
else:
    quality_report_before = pd.DataFrame({
        "ticker": pd.Series(dtype="string"),
        "issue_type": pd.Series(dtype="string"),
        "issue_date": pd.Series(dtype="datetime64[ns]"),
        "description": pd.Series(dtype="string"),
        "severity": pd.Series(dtype="string"),
    })

display(
    quality_report_before.groupby(["issue_type", "severity"])
    .size()
    .rename("issue_rows")
    .to_frame()
)

print("Total issue rows:", len(quality_report_before))
print(
    "Distinct affected observations:",
    len(quality_report_before[["ticker", "issue_date"]].drop_duplicates()),
)

,,issue_rows
issue_type,severity,
adjusted_price_inconsistency,high,10417
extreme_return,medium,5056
missing_price,high,18374
nonfinite_price,high,18374
nonpositive_price,high,5
open_below_low,high,5
zero_volume,medium,75


Total issue rows: 52306
Distinct affected observations: 33544


In [45]:
assert quality_report_before.columns.tolist() == [
    "ticker", "issue_type", "issue_date", "description", "severity"
]
assert quality_report_before["severity"].isin(
    ["high", "medium", "low"]
).all()

report_counts = quality_report_before["issue_type"].value_counts()

expected_counts = {
    **price_value_flags.sum().to_dict(),
    **ohlc_flags.sum().to_dict(),
    **volume_flags.sum().to_dict(),
    "extreme_return": int(return_review["extreme_return"].sum()),
    "adjusted_price_inconsistency": int(adjusted_disagrees.sum()),
    "missing_trading_date": len(missing_date_details),
}

for issue_type, expected_count in expected_counts.items():
    assert report_counts.get(issue_type, 0) == expected_count, issue_type

print("Quality-report schema and count checks passed.")

Quality-report schema and count checks passed.


## 11. Repair adjusted close and quarantine unresolved observations

Quarantine the five leading zero-open observations. Each also has an invalid opening price in the supplied reference, and no preceding observation is available for forward filling.

For retained observations, reconstruct missing or inconsistent adjusted close as validated raw close multiplied by the trusted same-date reference adjustment factor. Preserve original adjusted close, the factor, and separate repair flags. Leave consistent adjusted closes unchanged.

Retain statistical extreme-return and reported zero-volume flags for review. Neither flag alone triggers removal.

This function implements the evidence-based repairs needed by the actual delivery. Required behaviors absent from this delivery, including missing-price forward filling and missing-volume replacement, will be demonstrated separately on controlled examples.

In [46]:
def clean_actual_delivery(comparison, return_review):
    """Repair adjusted close and quarantine this delivery's leading zero opens."""
    candidates = comparison.copy()

    raw_columns = ["open", "high", "low", "close"]
    raw_prices = candidates[raw_columns]

    invalid_raw = (
        (~np.isfinite(raw_prices)).any(axis=1)
        | raw_prices.le(0).any(axis=1)
        | detect_ohlc_issues(candidates).any(axis=1)
    )

    first_dates = candidates.groupby("ticker")["ts"].transform("min")
    documented_zero_open = (
        candidates["open"].eq(0)
        & candidates["open_ref"].eq(0)
        & candidates["ts"].eq(first_dates)
    )

    # Stop if the delivery contains raw-price problems beyond our diagnosis.
    if not invalid_raw.equals(documented_zero_open):
        raise ValueError("Unexpected raw-price issue: investigate before repair.")

    quarantined = candidates.loc[invalid_raw].copy()
    quarantined["quarantine_reason"] = (
        "Leading zero open also present in reference; "
        "no preceding observation available for forward fill."
    )

    cleaned = candidates.loc[~invalid_raw].copy()
    cleaned["adj_close_original"] = cleaned["adj_close"]

    factor_valid = (
        np.isfinite(cleaned["reference_factor"])
        & cleaned["reference_factor"].gt(0)
    )
    if not factor_valid.all():
        raise ValueError("Unusable adjustment factor: investigate affected rows.")

    cleaned["adj_close_missing_repaired"] = cleaned["adj_close"].isna()

    cleaned["adj_close_inconsistent_repaired"] = (
        cleaned["adj_close"].notna()
        & ~np.isclose(
            cleaned["adj_close"],
            cleaned["close"] * cleaned["reference_factor"],
            rtol=1e-8,
            atol=1e-10,
        )
    )

    needs_repair = (
        cleaned["adj_close_missing_repaired"]
        | cleaned["adj_close_inconsistent_repaired"]
    )

    cleaned.loc[needs_repair, "adj_close"] = (
        cleaned.loc[needs_repair, "close"]
        * cleaned.loc[needs_repair, "reference_factor"]
    )

    # Join review flags by economic keys; merging can change row indices.
    cleaned = cleaned.merge(
        return_review[["ticker", "ts", "extreme_return"]],
        on=["ticker", "ts"],
        how="left",
        validate="one_to_one",
    )

    cleaned["zero_volume_review"] = cleaned["volume"].eq(0)
    cleaned["volume_missing_repaired"] = False

    return (
        cleaned.sort_values(["ticker", "ts"]).reset_index(drop=True),
        quarantined.sort_values(["ticker", "ts"]).reset_index(drop=True),
    )

In [47]:
cleaned, quarantined = clean_actual_delivery(comparison, return_review)

assert not detect_invalid_prices(cleaned).any().any()
assert not detect_ohlc_issues(cleaned).any().any()
assert np.isfinite(cleaned["volume"]).all()
assert cleaned["volume"].ge(0).all()
assert not cleaned.duplicated(["ticker", "ts"]).any()
assert cleaned["extreme_return"].notna().all()

assert np.isclose(
    cleaned["adj_close"],
    cleaned["close"] * cleaned["reference_factor"],
    rtol=1e-8,
    atol=1e-10,
).all()

# Confirm unaffected adjusted prices were preserved exactly.
repaired = (
    cleaned["adj_close_missing_repaired"]
    | cleaned["adj_close_inconsistent_repaired"]
)
assert cleaned.loc[~repaired, "adj_close"].equals(
    cleaned.loc[~repaired, "adj_close_original"]
)

# Every delivered row must have an accounted-for destination.
assert len(raw) == (
    int(exact_duplicate_mask.sum())
    + len(quarantined)
    + len(cleaned)
)

print("Quarantined observations:", len(quarantined))
print("Retained observations:", len(cleaned))
print(
    "Missing adjusted closes repaired:",
    cleaned["adj_close_missing_repaired"].sum(),
)
print(
    "Inconsistent adjusted closes repaired:",
    cleaned["adj_close_inconsistent_repaired"].sum(),
)
print("Post-repair validity and row-reconciliation checks passed.")

Quarantined observations: 5
Retained observations: 368246
Missing adjusted closes repaired: 18374
Inconsistent adjusted closes repaired: 10417
Post-repair validity and row-reconciliation checks passed.


**Repair results:** Quarantined five unresolved leading zero-open observations and retained 368,246 observations. Reconstructed 18,374 missing adjusted closes and 10,417 inconsistent adjusted closes using validated date-specific reference factors. Unaffected adjusted closes were preserved exactly.

Post-repair checks passed for finite positive prices, raw OHLC bounds, adjusted-price consistency, finite nonnegative volume, unique ticker-date keys, and complete row accounting. Statistical return flags and reported zero-volume flags remain review indicators.

These results concern the retained dataset. Quarantined observations remain separately available with reasons; the source database has not been modified.

In [48]:
summary_columns = ["open", "high", "low", "close", "adj_close", "volume"]

before_after_statistics = pd.concat(
    {
        "before_repair": deduplicated[summary_columns].describe(),
        "after_repair": cleaned[summary_columns].describe(),
    },
    names=["stage", "statistic"],
)

display(before_after_statistics)

row_accounting = pd.DataFrame({
    "category": [
        "Original delivered rows",
        "Exact duplicate copies removed",
        "Quarantined rows",
        "Retained cleaned rows",
    ],
    "rows": [
        len(raw),
        int(exact_duplicate_mask.sum()),
        len(quarantined),
        len(cleaned),
    ],
})

display(row_accounting)

open           high            low  \
stage         statistic                                                
before_repair count      368251.000000  368251.000000  368251.000000   
              mean           46.308766      46.771204      45.839768   
              std            67.143015      67.789068      66.489203   
              min             0.000000       0.005208       0.004801   
              25%             4.210000       4.250000       4.170000   
              50%            19.890000      20.090000      19.670000   
              75%            57.560000      58.160000      56.990000   
              max           604.260000     608.630000     597.880000   
after_repair  count      368246.000000  368246.000000  368246.000000   
              mean           46.309394      46.771824      45.840375   
              std            67.143254      67.789319      66.489450   
              min             0.005208       0.005208       0.004801   
              25%             4.210000       4.250000       4.170000   
              50%            19.890000      20.090000      19.670000   
              75%            57.560000      58.160000      56.997500   
              max           604.260000     608.630000     597.880000   

                                 close      adj_close        volume  
stage         statistic                                              
before_repair count      368251.000000  349877.000000  3.682510e+05  
              mean           46.316711      51.711536  3.095142e+07  
              std            67.154690     195.001654  1.266145e+08  
              min             0.005208       0.002945  0.000000e+00  
              25%             4.210000       1.395997  1.731400e+06  
              50%            19.900000      11.980000  4.649600e+06  
              75%            57.570000      43.790000  1.102151e+07  
              max           605.400000   10442.510783  9.276606e+09  
after_repair  count      368246.000000  368246.000000  3.682460e+05  
              mean           46.317325      37.274670  3.095184e+07  
              std            67.154939      63.587872  1.266153e+08  
              min             0.005208       0.002945  0.000000e+00  
              25%             4.210000       1.310000  1.731514e+06  
              50%            19.910000      11.460000  4.649648e+06  
              75%            57.570000      41.570000  1.102160e+07  
              max           605.400000     605.400000  9.276606e+09

,category,rows
0,Original delivered rows,397711
1,Exact duplicate copies removed,29460
2,Quarantined rows,5
3,Retained cleaned rows,368246


### Controlled missing-value repair example

Test the assignment’s missing-value rules on synthetic observations: remove rows with all four raw prices missing before filling; sort and forward-fill within each ticker; replace missing volume with zero while retaining a flag.

Recheck raw price validity and OHLC bounds after filling. Quarantine unresolved leading gaps and fills that create invalid ranges. This example tests raw-price and volume repair only; adjusted-close consistency is handled separately by the date-specific factor policy.

In [49]:
def repair_missing_raw_values(frame):
    """Repair missing raw prices/volume; separate dropped and unresolved rows."""
    data = frame.sort_values(["ticker", "ts"]).copy()
    raw_columns = ["open", "high", "low", "close"]

    all_raw_missing = data[raw_columns].isna().all(axis=1)
    removed = data.loc[all_raw_missing].copy()
    removed["removal_reason"] = "All four raw OHLC prices missing."

    data = data.loc[~all_raw_missing].copy()

    for column in raw_columns:
        was_missing = data[column].isna()
        data[column] = data.groupby("ticker")[column].ffill()
        data[f"{column}_forward_filled"] = (
            was_missing & data[column].notna()
        )

    data["volume_missing_repaired"] = data["volume"].isna()
    data["volume"] = data["volume"].fillna(0)

    raw_prices = data[raw_columns]
    unresolved = (
        (~np.isfinite(raw_prices)).any(axis=1)
        | raw_prices.le(0).any(axis=1)
        | detect_ohlc_issues(data).any(axis=1)
    )

    quarantined_example = data.loc[unresolved].copy()
    quarantined_example["quarantine_reason"] = (
        "Unresolved raw-price gap or invalid OHLC after forward fill."
    )

    return data.loc[~unresolved].copy(), removed, quarantined_example

In [50]:
repair_example = pd.DataFrame(
    [
        ["A", "2024-01-02", 100, 105, 95, 100, 10],
        ["A", "2024-01-03", 101, 106, 96, np.nan, np.nan],
        ["A", "2024-01-04", np.nan, np.nan, np.nan, np.nan, 20],
        ["A", "2024-01-05", 120, 125, 115, np.nan, 30],
        ["B", "2024-01-02", 50, 55, 45, np.nan, 10],
    ],
    columns=["ticker", "ts", "open", "high", "low", "close", "volume"],
)
repair_example["ts"] = pd.to_datetime(repair_example["ts"])

# Shuffle the input to verify that the function sorts before filling.
example_clean, example_removed, example_quarantine = (
    repair_missing_raw_values(
        repair_example.sample(frac=1, random_state=42)
    )
)

assert len(example_clean) == 2
assert set(example_removed.index) == {2}
assert set(example_quarantine.index) == {3, 4}

assert example_clean.loc[1, "close"] == 100
assert example_clean.loc[1, "close_forward_filled"]
assert example_clean.loc[1, "volume"] == 0
assert example_clean.loc[1, "volume_missing_repaired"]

# B's leading gap must not borrow a value from ticker A.
assert pd.isna(example_quarantine.loc[4, "close"])

# A's filled close of 100 violates its later daily low of 115.
assert example_quarantine.loc[3, "close"] == 100

print("Missing-price and volume repair tests passed.")

Missing-price and volume repair tests passed.


### Controlled interpolation of confirmed errors

Demonstrate interpolation only for observations explicitly identified as erroneous in a synthetic example. Interpolate within each ticker after sorting, using valid observations on both sides. Do not extrapolate leading or trailing errors or borrow another ticker’s prices.

The interpolation is linear by observation position. It uses future observations and is therefore a retrospective repair, not a method available in real time.


In [51]:
def interpolate_confirmed_close_errors(frame):
    """Interpolate confirmed close errors within ticker; retain unresolved edges."""
    data = frame.sort_values(["ticker", "ts"]).copy()
    data["close_original"] = data["close"]

    # Only explicitly confirmed errors are eligible for replacement.
    data.loc[data["confirmed_close_error"], "close"] = np.nan

    interpolated = data.groupby("ticker")["close"].transform(
        lambda values: values.interpolate(
            method="linear",
            limit_area="inside",
        )
    )

    repaired = (
        data["confirmed_close_error"]
        & interpolated.notna()
    )
    data.loc[repaired, "close"] = interpolated.loc[repaired]
    data["close_interpolated"] = repaired

    return data


interpolation_example = pd.DataFrame({
    "ticker": ["A"] * 5 + ["B"] * 3,
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04",
        "2024-01-05", "2024-01-08",
        "2024-01-02", "2024-01-03", "2024-01-04",
    ]),
    "close": [999, 100, 999, 104, 999, 50, 999, 54],
    "confirmed_close_error": [
        True, False, True, False, True, False, True, False
    ],
})

interpolation_result = interpolate_confirmed_close_errors(
    interpolation_example.sample(frac=1, random_state=42)
)

assert interpolation_result.loc[2, "close"] == 102
assert interpolation_result.loc[6, "close"] == 52
assert pd.isna(interpolation_result.loc[0, "close"])
assert pd.isna(interpolation_result.loc[4, "close"])
assert interpolation_result["close_interpolated"].sum() == 2

unchanged = ~interpolation_result["confirmed_close_error"]
# Missing-value handling can change dtype; verify exact numeric preservation.
np.testing.assert_array_equal(
    interpolation_result.loc[unchanged, "close"].to_numpy(),
    interpolation_result.loc[unchanged, "close_original"].to_numpy(),
)

print("Confirmed-error interpolation tests passed.")
display(interpolation_result)

Confirmed-error interpolation tests passed.


,ticker,ts,close,confirmed_close_error,close_original,close_interpolated
0,A,2024-01-02,NaN,True,999,False
1,A,2024-01-03,100.0,False,100,False
2,A,2024-01-04,102.0,True,999,True
3,A,2024-01-05,104.0,False,104,False
4,A,2024-01-08,NaN,True,999,False
5,B,2024-01-02,50.0,False,50,False
6,B,2024-01-03,52.0,True,999,True
7,B,2024-01-04,54.0,False,54,False


In [52]:
# Verify the interpolated closes against full OHLC bounds and same-date factors.
interpolation_validation = interpolation_result.copy()
is_a = interpolation_validation["ticker"].eq("A")
interpolation_validation["open"] = np.where(is_a, 100.0, 50.0)
interpolation_validation["low"] = np.where(is_a, 95.0, 45.0)
interpolation_validation["high"] = np.where(is_a, 110.0, 60.0)

# Known synthetic factors vary by date; never forward-fill across this change.
factors_by_row = pd.Series({0: 0.5, 1: 0.5, 2: 0.25, 3: 0.25,
                            4: 0.25, 5: 1.0, 6: 1.0, 7: 1.0})
interpolation_validation["reference_factor"] = factors_by_row
interpolation_validation["adj_close"] = (
    interpolation_validation["close"]
    * interpolation_validation["reference_factor"]
)
unresolved_edges = interpolation_validation["close"].isna()
assert set(interpolation_validation.index[unresolved_edges]) == {0, 4}
validated_example = interpolation_validation.loc[~unresolved_edges]
assert not detect_invalid_prices(validated_example).any().any()
assert not detect_ohlc_issues(validated_example).any().any()
assert validated_example.loc[2, "adj_close"] == 25.5
assert validated_example.loc[6, "adj_close"] == 52.0

# A numerically interpolated close must still be rejected if it violates OHLC.
invalid_range_example = validated_example.copy()
invalid_range_example.loc[2, "high"] = 101.0
assert detect_ohlc_issues(invalid_range_example).loc[2, "close_above_high"]
print("Interpolation bounds, edge quarantine, and adjustment checks passed.")


Interpolation bounds, edge quarantine, and adjustment checks passed.


## 12. Revalidate the retained dataset

Rerun price, OHLC, volume, adjustment-consistency, return-outlier, and reference-calendar checks after repair.

Recalculate statistical return flags because removing observations changes the retained history. Preserve the original diagnosis separately.

Quarantine creates five expected coverage gaps relative to the supplied reference. Report these as explicit exclusions with reasons rather than treating the retained dataset as having complete reference coverage. Reported zero volume and extreme returns may remain as review flags.

In [53]:
after_parts = []

report_columns = [
    "ticker",
    "issue_type",
    "issue_date",
    "description",
    "severity",
]

after_price_flags = detect_invalid_prices(cleaned)
after_ohlc_flags = detect_ohlc_issues(cleaned)
after_volume_flags = detect_volume_issues(cleaned)

for flag_table in [after_price_flags, after_ohlc_flags, after_volume_flags]:
    for issue_type in flag_table.columns:
        after_parts.append(issue_rows(
            cleaned,
            flag_table[issue_type],
            issue_type,
            f"Post-repair check: {issue_type}.",
            "medium" if issue_type == "zero_volume" else "high",
        ))

after_adjustment_issue = ~np.isclose(
    cleaned["adj_close"],
    cleaned["close"] * cleaned["reference_factor"],
    rtol=1e-8,
    atol=1e-10,
)

after_parts.append(issue_rows(
    cleaned,
    after_adjustment_issue,
    "adjusted_price_inconsistency",
    "Adjusted close is inconsistent with the trusted date-specific factor.",
    "high",
))

# Preserve the original flags only once, including when this cell is rerun.
if "extreme_return_before_cleaning" not in cleaned.columns:
    cleaned["extreme_return_before_cleaning"] = cleaned["extreme_return"].copy()

after_returns = cleaned.groupby("ticker")["close"].pct_change(
    fill_method=None
)
cleaned["extreme_return"] = (
    after_returns.groupby(cleaned["ticker"])
    .transform(flag_extreme_returns)
)

after_parts.append(issue_rows(
    cleaned,
    cleaned["extreme_return"],
    "extreme_return",
    "Retained-history simple return exceeds its ticker's 3-sigma threshold.",
    "medium",
))

quarantine_reasons = quarantined.set_index(
    ["ticker", "ts"]
)["quarantine_reason"].to_dict()

after_gap_rows = []

for ticker, reference_group in reference_calendar.groupby("ticker"):
    gaps = find_missing_dates(
        cleaned.loc[cleaned["ticker"].eq(ticker), "ts"],
        reference_group["ts"],
    )

    for date in gaps:
        reason = quarantine_reasons.get((ticker, date))
        after_gap_rows.append({
            "ticker": ticker,
            "issue_type": "missing_trading_date",
            "issue_date": date,
            "description": (
                f"Excluded by quarantine: {reason}"
                if reason is not None
                else "Absent from retained data relative to reference."
            ),
            "severity": "medium",
        })

after_parts.append(pd.DataFrame(after_gap_rows, columns=report_columns))

nonempty_after = [part for part in after_parts if not part.empty]

quality_report_after = (
    pd.concat(nonempty_after, ignore_index=True)[report_columns]
    .sort_values(["ticker", "issue_date", "issue_type"])
    .reset_index(drop=True)
    if nonempty_after
    else quality_report_before.iloc[:0].copy()
)

display(
    quality_report_after.groupby(["issue_type", "severity"])
    .size()
    .rename("issue_rows")
    .to_frame()
)

,,issue_rows
issue_type,severity,
extreme_return,medium,5057
missing_trading_date,medium,5
zero_volume,medium,74


In [54]:
assert not quality_report_after["severity"].eq("high").any()

reported_gaps = {
    (row["ticker"], row["issue_date"])
    for row in after_gap_rows
}
quarantined_keys = set(
    quarantined[["ticker", "ts"]].itertuples(index=False, name=None)
)

assert reported_gaps == quarantined_keys
assert len(reported_gaps) == 5

issue_comparison = pd.concat(
    [
        quality_report_before["issue_type"].value_counts().rename("before"),
        quality_report_after["issue_type"].value_counts().rename("after"),
    ],
    axis=1,
).fillna(0).astype(int)

display(issue_comparison)
print("Post-repair quality-report checks passed.")

,before,after
issue_type,,
missing_price,18374,0
nonfinite_price,18374,0
adjusted_price_inconsistency,10417,0
extreme_return,5056,5057
zero_volume,75,74
nonpositive_price,5,0
open_below_low,5,0
missing_trading_date,0,5


Post-repair quality-report checks passed.


### Post-repair validation findings

No missing, non-finite, nonpositive, OHLC-inconsistent, or adjustment-inconsistent prices remain in the retained dataset.

The five reference-calendar gaps correspond exactly to quarantined observations. Reported zero-volume flags decreased from 75 to 74 following quarantine.

The recalculated extreme-return count increased from 5,056 to 5,057. Because this screen depends on the retained history’s return distribution, removing observations can change classifications. These statistical flags remain review indicators rather than confirmed errors.

In [55]:
# Recover original flags explicitly by key, independent of prior cell reruns.
original_flags = return_review[
    ["ticker", "ts", "extreme_return"]
].rename(columns={"extreme_return": "original_extreme_return"})

flag_comparison = cleaned[
    ["ticker", "ts", "extreme_return"]
].merge(
    original_flags,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one",
)

changed_flags = flag_comparison.loc[
    flag_comparison["extreme_return"]
    .ne(flag_comparison["original_extreme_return"])
]

display(changed_flags)

assert quarantined["volume"].eq(0).sum() == 1
print("Zero-volume count change reconciled to quarantine.")

,ticker,ts,extreme_return,original_extreme_return
219181,MCD,1989-06-21,True,False


Zero-volume count change reconciled to quarantine.


## 13. Persist the cleaned dataset and audit outputs

Create a separate homework database at `data/data.db` from the course source using SQLite’s backup operation. Write `ohlc_hw_cleaned` only to this working database.

Store observation dates as ISO strings (`YYYY-MM-DD`) so later SQL date filters behave consistently. Retain original ticker labels, original adjusted close, adjustment factors, and repair/review flags.

Save diagnosis, quarantine, normalization, and reconciliation tables as local audit outputs. Verify the persisted cleaned table by reading it back. Database files are local working artifacts and must not be committed.

In [56]:
homework_dir = Path.cwd()
working_db = homework_dir / "data" / "data.db"
audit_dir = homework_dir / "audit"

working_db.parent.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)

if working_db.resolve() == source_db.resolve():
    raise ValueError("Working database must differ from the source database.")

# Create the working copy once; do not overwrite an existing database.
if not working_db.exists():
    with closing(sqlite3.connect(source_uri, uri=True)) as source_conn:
        with closing(sqlite3.connect(working_db)) as destination_conn:
            source_conn.backup(destination_conn)

persist_columns = [
    "ts", "ticker", "open", "high", "low", "close", "adj_close", "volume",
    "ticker_raw", "adj_close_original", "reference_factor",
    "adj_close_missing_repaired", "adj_close_inconsistent_repaired",
    "volume_missing_repaired", "zero_volume_review", "extreme_return",
]

persisted = cleaned[persist_columns].copy()
persisted["ts"] = persisted["ts"].dt.strftime("%Y-%m-%d")

with closing(sqlite3.connect(working_db)) as conn:
    persisted.to_sql(
        "ohlc_hw_cleaned",
        conn,
        if_exists="replace",
        index=False,
        chunksize=10_000,
    )

quality_report_before.to_csv(audit_dir / "quality_report_before.csv", index=False)
quality_report_after.to_csv(audit_dir / "quality_report_after.csv", index=False)
quarantined.to_csv(audit_dir / "quarantined.csv", index=False)
ticker_audit.to_csv(audit_dir / "ticker_normalization.csv", index=False)
row_accounting.to_csv(audit_dir / "row_accounting.csv", index=False)

print("Working database:", working_db)
print("Saved cleaned rows:", len(persisted))

Working database: C:\Users\scouts.admin\Documents\UCB-MFE-python-preprogram-2026-winter\Homeworks\HW2\Sterling_Couts\work\final_verification\data\data.db
Saved cleaned rows: 368246


In [57]:
with closing(sqlite3.connect(
    working_db.resolve().as_uri() + "?mode=ro", uri=True
)) as conn:
    saved = pd.read_sql_query(
        "SELECT * FROM ohlc_hw_cleaned ORDER BY ticker, ts",
        conn,
        parse_dates=["ts"],
    )
    raw_count_in_copy = pd.read_sql_query(
        "SELECT COUNT(*) AS n FROM ohlc_hw", conn
    ).loc[0, "n"]

assert len(saved) == len(cleaned)
assert raw_count_in_copy == len(raw)
assert not saved.duplicated(["ticker", "ts"]).any()
assert not detect_invalid_prices(saved).any().any()
assert not detect_ohlc_issues(saved).any().any()
assert np.isfinite(saved["volume"]).all()
assert saved["volume"].ge(0).all()

np.testing.assert_allclose(
    saved["adj_close"],
    saved["close"] * saved["reference_factor"],
    rtol=1e-8,
    atol=1e-10,
)

print("Saved-table validation passed.")

Saved-table validation passed.


### Persistence verification

Saved 368,246 observations to `ohlc_hw_cleaned` in the separate homework database, `data/data.db`. Read-back checks passed for row count, unique ticker-date keys, price validity, OHLC bounds, adjustment consistency, and finite nonnegative volume. The working copy retains all 397,711 original homework rows in `ohlc_hw`.

The sole changed extreme-return classification is MCD on June 21, 1989, which becomes flagged after recalculation on the retained history. Its price was not repaired. The zero-volume count reduction is accounted for by one quarantined observation.

Pre/post quality reports, quarantine reasons, ticker normalization, and row accounting were saved as local audit outputs.